# 섹션3-9. AI를 활용한 HeatMap

> 강의: [32가지 데이터 시각화 전략 - 비전공자를 위한 기초이론 & 실습](https://www.inflearn.com/course/32-data-visualizatio/dashboard?cid=343563) (반병현) — 전체 20강

- [x] 강의 시청 완료
- [x] 실습/정리 완료

## 배운 내용

<!-- 강의를 보면서 핵심을 적는다 -->

-

## 목표 / 재현할 것

<!-- 이 강의에서 만든 차트를 내 방식대로 다시 만들어본다 -->

-


## 실습

강의 예시(요일 x 시간대 방문객 수)는 원본 데이터가 없어, 같은 구조(범주 x 범주 격자에
값 하나)를 두 가지 실측 데이터로 재현한다 — ①은 강의처럼 진짜 주기 패턴이 있는 경우,
②는 패턴처럼 보이지만 실제로는 노이즈인 경우. 강의가 안 다룬 함정까지 확인한다.

In [ ]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from viz_utils import setup, load_sample

setup()
rng = np.random.default_rng(0)

sensor = load_sample("sensor")  # 10분 간격 2주, 요일·시간대가 있는 센서 데이터

### 1. 그냥 그리면 — 강의가 말한 '워라소모금' 상태

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(sensor.index, sensor["온도"], lw=0.6, color="#4C78A8")
ax.set_title("온도 원본 시계열 — 반복되는 패턴이 있는 건 알겠는데 감이 안 온다")
plt.show()

### 2. 히트맵으로 — 요일 x 시간대

In [ ]:
pivot = sensor["온도"].groupby([sensor.index.dayofweek, sensor.index.hour]).mean().unstack()
pivot.index = ["월", "화", "수", "목", "금", "토", "일"]

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
ax.set_yticks(range(7), pivot.index)
ax.set_xticks(range(0, 24, 2), range(0, 24, 2))
ax.set(xlabel="시(hour)", title="요일 x 시간대 평균 온도 — 노란색일수록 높다")
fig.colorbar(im, ax=ax, label="온도(°C)")
plt.show()

> 강의 설명 그대로다 — "요일별로 시간대별로 찐한 영역일수록 값이 많은 것"이 한눈에
> 보인다. 이 센서 데이터는 실제로 하루 주기(낮에 덥고 밤에 서늘함)를 심어서 만든 것이라
> 히트맵의 색 차이가 **진짜 패턴**이다. 매니저라면 "몇 시부터 인력을 더 투입해야 하는가"에
> 해당하는 질문에 바로 답할 수 있다.

### 3. 함정 — 패턴처럼 보이지만 노이즈일 수 있다

In [ ]:
apple = pd.read_csv("../references/lecture_data/애플 주가/애플 주가.csv")
apple["Date"] = pd.to_datetime(apple["Date"])
monthly = apple.set_index("Date")["Close"].resample("ME").last()
ret = monthly.pct_change().dropna() * 100

t = pd.DataFrame({"연": ret.index.year, "월": ret.index.month, "수익률": ret.values})
heat = t.pivot(index="연", columns="월", values="수익률")

fig, ax = plt.subplots(figsize=(10, 4.5))
vmax = heat.abs().max().max()
im = ax.imshow(heat.values, cmap="RdBu", vmin=-vmax, vmax=vmax, aspect="auto")  # 발산형 컬러맵, 0 중심 고정
ax.set_yticks(range(len(heat.index)), heat.index)
ax.set_xticks(range(12), range(1, 13))
ax.set(xlabel="월", title="애플 주가 월별 수익률(%) — 계절 패턴처럼 보인다")
fig.colorbar(im, ax=ax, label="월 수익률(%)")
plt.show()

### 4. 진짜 패턴인지 검증 — 순열검정

In [ ]:
monthly_mean = t.groupby("월")["수익률"].mean()
observed_spread = monthly_mean.max() - monthly_mean.min()
print(f"월평균 최고 {monthly_mean.idxmax()}월 {monthly_mean.max():.2f}%, "
      f"최저 {monthly_mean.idxmin()}월 {monthly_mean.min():.2f}% → 폭 {observed_spread:.2f}%p")

n_perm = 20000
spreads = np.empty(n_perm)
values, labels = t["수익률"].values, t["월"].values
for i in range(n_perm):
    shuffled = rng.permutation(values)
    gm = pd.Series(shuffled).groupby(labels).mean()
    spreads[i] = gm.max() - gm.min()

p = (spreads >= observed_spread).mean()

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(spreads, bins=30, color="#4C78A8", alpha=0.7)
ax.axvline(observed_spread, color="#E45756", ls="--", lw=2, label="관측값")
ax.set(xlabel="월 라벨을 무작위로 섞었을 때의 (최고-최저) 폭", ylabel="빈도")
ax.set_title(f"순열검정 {n_perm:,}회 — p = {p:.3f}")
ax.legend()
plt.show()

print(f"월 라벨을 무작위로 섞어도 이만한 폭이 나오는 비율: {p:.1%}")

**결론.** n=10(연도 수)으로 월별 평균을 내면 우연히도 큰 폭이 자주 나온다
(순열검정 p≈0.49 — 거의 절반). 히트맵의 색 차이가 실제 계절성인지 노이즈인지는
**히트맵만 봐서는 구별할 수 없다.** 강의의 방문객 히트맵은 표본이 충분해(각 칸마다
여러 주의 관측치) 패턴을 믿을 만하지만, 표본이 적은 히트맵(연도별 x 월별처럼 칸당
관측치가 하나뿐인 경우)은 색이 진하다고 곧바로 의미로 해석하면 안 된다.

### 정리 — 히트맵을 믿어도 되는 조건

| | 요일 x 시간대 (강의 예시) | 연도 x 월 (애플 주가) |
| --- | --- | --- |
| 칸당 관측치 | 여러 주 반복 관측 | 연 1회뿐(n=10) |
| 순열검정 | (해보면 유의할 가능성 높음 — 물리적 원인 있음) | p ≈ 0.49, 노이즈와 구별 안 됨 |
| 결론 | 색 차이를 믿고 의사결정에 반영 가능 | 색 차이만으로 계절성을 주장하면 안 됨 |


---

## 메모

-
